<a href="https://colab.research.google.com/github/hamdikhasawneh/AI-sepsis/blob/main/notebooks/01_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DATA_DIR   = Path('/content/drive/MyDrive/gp/Cleaned')
OUTPUT_DIR = Path('/content/drive/MyDrive/mimic_iv_processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR exists:", DATA_DIR.exists())
print("OUTPUT_DIR exists:", OUTPUT_DIR.exists())

DATA_DIR exists: True
OUTPUT_DIR exists: True


In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/gp/Cleaned/chartevents_labevents.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist())

['Cleaned/', 'Cleaned/chartevents.csv', 'Cleaned/labevents.csv']


In [ ]:
for file in DATA_DIR.iterdir():
    print(file.name)

inputevents.csv
outputevents.csv
procedureevents.csv
ingredientevents.csv
datetimeevents.csv
icustays.csv
d_items.csv
patients.csv
prescriptions.csv
admissions.csv
microbiologyevents.csv
caregiver.csv
d_labitems.csv
chartevents_labevents.zip
icu_cohort.csv


##Load base tables


In [ ]:
import pandas as pd

In [ ]:
patients = pd.read_csv(DATA_DIR / 'patients.csv')
admissions = pd.read_csv(DATA_DIR / 'admissions.csv')
icustays = pd.read_csv(DATA_DIR / 'icustays.csv')

## Inspect Tables

In this section, we check:
- shape
- columns
- sample rows


In [ ]:
print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("icustays:", icustays.shape)

patients: (364627, 6)
admissions: (546028, 16)
icustays: (94458, 8)


In [ ]:
print("patients columns:")
print(patients.columns.tolist())
print("\nadmissions columns:")
print(admissions.columns.tolist())
print("\nicustays columns:")
print(icustays.columns.tolist())

patients columns:
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']

admissions columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']

icustays columns:
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']


In [ ]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [ ]:
admissions.head(10)
admissions[admissions.subject_id == 10000032]

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0


In [ ]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


## Creating Cohort


In [ ]:
cohort = icustays.merge(
    admissions,
    on=['subject_id', 'hadm_id'],
    how='left'
).merge(
    patients,
    on='subject_id',
    how='left'
)
cohort.head()
print("cohort shape:", cohort.shape)

cohort shape: (94458, 27)


In [ ]:
cohort.isnull().sum().sort_values(ascending=False).head(20)

,0
deathtime,83117
dod,56491
edregtime,32331
edouttime,32331
marital_status,7761
insurance,1523
discharge_location,848
language,396
outtime,14
los,14


In [ ]:
print("Duplicate rows:", cohort.duplicated().sum())

Duplicate rows: 0


### Saving Cohort

In [ ]:
output_dir = Path('/content/drive/MyDrive/mimic_iv_processed')
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
print("Cohort shape:", cohort.shape)
print(cohort["stay_id"].nunique())
print(cohort.shape[0])
print(cohort.duplicated(subset="stay_id").sum())

Cohort shape: (94458, 27)
94458
94458
0


In [ ]:
cohort.columns

Index(['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit',
       'intime', 'outtime', 'los', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'gender',
       'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype='object')

### Convert Data-Time Columns



In [ ]:
cohort["intime"] = pd.to_datetime(cohort["intime"])
cohort["outtime"] = pd.to_datetime(cohort["outtime"])
cohort["admittime"] = pd.to_datetime(cohort["admittime"])
cohort["dischtime"] = pd.to_datetime(cohort["dischtime"])

#### Calculate ICU Length of Stay


In [ ]:
cohort["icu_los_hours"] = (cohort["outtime"] - cohort["intime"]).dt.total_seconds() / 3600
cohort["icu_los_hours"].describe()

,icu_los_hours
count,94444.000000
mean,87.120596
std,129.659365
min,0.030000
25%,26.309097
50%,47.175556
75%,92.701806
max,5433.673889


### Filtering Short ICU stays

In [ ]:
# Removing less than 4 hours
cohort = cohort[cohort["icu_los_hours"] >= 4]
cohort.shape

(93730, 28)

In [ ]:
cohort["anchor_age"].describe()

,anchor_age
count,93730.000000
mean,63.045610
std,16.712152
min,18.000000
25%,53.000000
50%,65.000000
75%,76.000000
max,91.000000


In [ ]:
# Filtering adult patients
cohort = cohort[cohort["anchor_age"] >= 18]
print(cohort.shape)

(93730, 28)


In [ ]:
# NEW: drop duplicate stay_ids if any exist
cohort = cohort.drop_duplicates(subset='stay_id')
print("Final cohort shape:", cohort.shape)
print("Unique stays:", cohort['stay_id'].nunique())
# NEW: create subject_id → stay_id mapping for later split
# We split by subject_id to prevent patient leakage across train/test
print("Unique patients:", cohort['subject_id'].nunique())

Final cohort shape: (93730, 28)
Unique stays: 93730
Unique patients: 65241


In [ ]:
# saved new file called icu_cohort.csv
output_dir = Path('/content/drive/MyDrive/mimic_iv_processed')
output_dir.mkdir(parents=True, exist_ok=True)

cohort.to_csv(output_dir / "icu_cohort.csv", index=False)

print("Clean ICU cohort saved")

Clean ICU cohort saved


## Identify Vital Sign ITEMIDs

In [ ]:
d_items = pd.read_csv(DATA_DIR / "d_items.csv")
print(d_items.shape)
print(d_items.columns)

(4095, 9)
Index(['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname',
       'param_type', 'lownormalvalue', 'highnormalvalue'],
      dtype='object')


In [ ]:
# Important Vital Signs
# Heart Rate
d_items[d_items["label"].str.contains("Heart Rate", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [ ]:
# Respiratory Rate
d_items[d_items["label"].str.contains("Respiratory", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
28,220210,Respiratory Rate,RR,chartevents,Respiratory,insp/min,Numeric,NaN,NaN
470,223985,Respiratory Pattern,Respiratory Pattern,chartevents,Pulmonary,NaN,Text,NaN,NaN
475,223990,Respiratory Effort,Respiratory Effort,chartevents,Pulmonary,NaN,Text,NaN,NaN
799,224688,Respiratory Rate (Set),Respiratory Rate (Set),chartevents,Respiratory,insp/min,Numeric,NaN,NaN
800,224689,Respiratory Rate (spontaneous),Respiratory Rate (spontaneous),chartevents,Respiratory,insp/min,Numeric,NaN,NaN
801,224690,Respiratory Rate (Total),Respiratory Rate (Total),chartevents,Respiratory,insp/min,Numeric,NaN,36.0
839,224745,Respiratory Quotient,RQ,chartevents,Respiratory,NaN,Numeric,NaN,NaN
1327,225475,Respiratory Arrest,Respiratory Arrest,procedureevents,3-Significant Events,NaN,Processes,NaN,NaN
2075,227032,Non-Operative Respiratory (RESPIRAT),Respiratory Non-Operative,chartevents,Scores - APACHE IV,NaN,Text,NaN,NaN
2090,227047,Post-Operative Respiratory (RESPIRAT),Respiratory Post-Operative,chartevents,Scores - APACHE IV,NaN,Text,NaN,NaN


In [ ]:
# Temperature
d_items[d_items["label"].str.contains("Temperature", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
337,223761,Temperature Fahrenheit,Temperature F,chartevents,Routine Vital Signs,°F,Numeric,NaN,NaN
338,223762,Temperature Celsius,Temperature C,chartevents,Routine Vital Signs,°C,Numeric,NaN,NaN
505,224027,Skin Temperature,Skin Temp,chartevents,Skin - Assessment,NaN,Text,NaN,NaN
767,224642,Temperature Site,Temp Site,chartevents,Routine Vital Signs,NaN,Text,NaN,NaN
790,224674,Changes in Temperature,Changes in Temperature,chartevents,Toxicology,NaN,Text,NaN,NaN
1814,226329,Blood Temperature CCO (C),Blood Temp CCO (C),chartevents,Routine Vital Signs,°C,Numeric,NaN,NaN
2097,227054,TemperatureF_ApacheIV,TemperatureF_ApacheIV,chartevents,Scores - APACHE IV (2),°F,Numeric,NaN,NaN
2776,228242,Pt. Temperature (BG) (SOFT),Pt. Temperature (BG) (SOFT),chartevents,Labs,NaN,Numeric,NaN,NaN
3466,229236,Cerebral Temperature (C),Cerebral T (C),chartevents,Hemodynamics,°C,Numeric,NaN,NaN


In [ ]:
# SpO2
d_items[d_items["label"].str.contains("SpO2", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
1810,226253,SpO2 Desat Limit,SpO2 Desat Limit,chartevents,Alarms,%,Numeric,NaN,NaN
3920,229862,Forehead SpO2 Sensor in Place,Forehead SpO2 Sensor in Place,chartevents,Routine Vital Signs,NaN,Checkbox,NaN,NaN


In [ ]:
# Blood Pressure
d_items[d_items["label"].str.contains("Pressure", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
6,220050,Arterial Blood Pressure systolic,ABPs,chartevents,Routine Vital Signs,mmHg,Numeric,90.0,140.0
7,220051,Arterial Blood Pressure diastolic,ABPd,chartevents,Routine Vital Signs,mmHg,Numeric,60.0,90.0
8,220052,Arterial Blood Pressure mean,ABPm,chartevents,Routine Vital Signs,mmHg,Numeric,NaN,NaN
9,220056,Arterial Blood Pressure Alarm - Low,ABP Alarm - Low,chartevents,Alarms,mmHg,Numeric,NaN,NaN
10,220058,Arterial Blood Pressure Alarm - High,ABP Alarm - High,chartevents,Alarms,mmHg,Numeric,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3947,229892,Pressure Ulcer #9- Date Indentified,Pressure Ulcer #9- Date Indentified,datetimeevents,Skin - Impairment,NaN,Date and time,NaN,NaN
3948,229893,Pressure Ulcer #10- Date Indentified,Pressure Ulcer #10- Date Indentified,datetimeevents,Skin - Impairment,NaN,Date and time,NaN,NaN
3954,229899,Left Ventricular Pressure Signal - Systolic,Left Ventricular Pressure Signal - Systolic,chartevents,Impella,mmHg,Numeric,NaN,NaN
3955,229900,Left Ventricular Pressure Signal - Diastolic,Left Ventricular Pressure Signal - Diastolic,chartevents,Impella,mmHg,Numeric,NaN,NaN


In [ ]:
d_items[d_items["label"].str.contains("saturation", case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
31,220227,Arterial O2 Saturation,SaO2,chartevents,Labs,%,Numeric,NaN,NaN
36,220277,O2 saturation pulseoxymetry,SpO2,chartevents,Respiratory,%,Numeric,NaN,NaN
345,223769,O2 Saturation Pulseoxymetry Alarm - High,SpO2 Alarm - High,chartevents,Alarms,%,Numeric,NaN,NaN
346,223770,O2 Saturation Pulseoxymetry Alarm - Low,SpO2 Alarm - Low,chartevents,Alarms,%,Numeric,NaN,NaN
2004,226860,RA %O2 Saturation (PA Line),RA %O2 Saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2005,226861,ART %O2 saturation (PA Line),ART %O2 saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2006,226862,PA %O2 Saturation (PA Line),PA %O2 Saturation (PA Line),chartevents,PA Line Insertion,%,Numeric,NaN,NaN
2007,226863,PVR %O2 Saturation (PA Line),PVR %O2 Saturation (PA Line),chartevents,PA Line Insertion,dynes*sec/cm5,Numeric,NaN,NaN
2008,226865,SVR %O2 Saturation (PA Line),SVR %O2 Saturation (PA Line),chartevents,PA Line Insertion,dynes*sec/cm5,Numeric,NaN,NaN
2769,228232,PAR-Oxygen saturation,PAR-Oxygen saturation,chartevents,Routine Vital Signs,NaN,Text,NaN,NaN


In [ ]:
# Selected vital sign ITEMIDs

HEART_RATE_ID = 220045
RESP_RATE_ID = 220210
TEMP_C_ID = 223762
SPO2_ID = 220277
ABP_SYS_ID = 220050
ABP_DIA_ID = 220051
ABP_MEAN_ID = 220052
VITAL_ITEMIDS = [
    HEART_RATE_ID,
    RESP_RATE_ID,
    TEMP_C_ID,
    SPO2_ID,
    ABP_SYS_ID,
    ABP_DIA_ID,
    ABP_MEAN_ID
]
print(VITAL_ITEMIDS)

[220045, 220210, 223762, 220277, 220050, 220051, 220052]


## Extract Vital Signs from chartevents


In [ ]:
chartevents_zip_path = DATA_DIR / "chartevents_labevents.zip"
print(chartevents_zip_path.exists())

True


In [ ]:
import zipfile

with zipfile.ZipFile(chartevents_zip_path, 'r') as z:
    print(z.namelist())

['Cleaned/', 'Cleaned/chartevents.csv', 'Cleaned/labevents.csv']


## Save file vitals filtered

In [ ]:
import zipfile
import pandas as pd

chartevents_zip_path = DATA_DIR / "chartevents_labevents.zip"
output_file = output_dir / "vitals_filtered.csv"

stay_ids = set(cohort["stay_id"].dropna().unique())

VITAL_ITEMIDS = [
    220045,  # Heart Rate
    220210,  # Respiratory Rate
    223762,  # Temperature Celsius
    220277,  # O2 saturation pulseoxymetry
    220050,  # Arterial Blood Pressure systolic
    220051,  # Arterial Blood Pressure diastolic
    220052   # Arterial Blood Pressure mean
]

if output_file.exists():
    output_file.unlink()

chunk_size = 100000
first_chunk = True

with zipfile.ZipFile(chartevents_zip_path, "r") as z:
    with z.open("Cleaned/chartevents.csv") as f:
        for i, chunk in enumerate(
            pd.read_csv(
                f,
                chunksize=chunk_size,
                usecols=[
                    "subject_id",
                    "hadm_id",
                    "stay_id",
                    "charttime",
                    "itemid",
                    "valuenum",
                    "valueuom"
                ]
            )
        ):
            chunk = chunk[
                (chunk["stay_id"].isin(stay_ids)) &
                (chunk["itemid"].isin(VITAL_ITEMIDS))
            ]

            if not chunk.empty:
                chunk.to_csv(output_file, mode="a", header=first_chunk, index=False)
                first_chunk = False

            print(f"Finished chunk {i+1}")

print("Filtered vitals saved successfully")

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11
Finished chunk 12
Finished chunk 13
Finished chunk 14
Finished chunk 15
Finished chunk 16
Finished chunk 17
Finished chunk 18
Finished chunk 19
Finished chunk 20
Finished chunk 21
Finished chunk 22
Finished chunk 23
Finished chunk 24
Finished chunk 25
Finished chunk 26
Finished chunk 27
Finished chunk 28
Finished chunk 29
Finished chunk 30
Finished chunk 31
Finished chunk 32
Finished chunk 33
Finished chunk 34
Finished chunk 35
Finished chunk 36
Finished chunk 37
Finished chunk 38
Finished chunk 39
Finished chunk 40
Finished chunk 41
Finished chunk 42
Finished chunk 43
Finished chunk 44
Finished chunk 45
Finished chunk 46
Finished chunk 47
Finished chunk 48
Finished chunk 49
Finished chunk 50
Finished chunk 51
Finished chunk 52
Finished chunk 53
Finished chunk 54
Finished chunk 55
Finished chunk 56
F

In [ ]:
vitals = pd.read_csv(output_file)
print(vitals.shape)

(35596303, 7)


## Building LSTM


In [ ]:
itemid_to_label = {
    220045: "heart_rate",
    220210: "resp_rate",
    223762: "temp_c",
    220277: "spo2",
    220050: "abp_sys",
    220051: "abp_dia",
    220052: "abp_mean"
}

vitals["feature_name"] = vitals["itemid"].map(itemid_to_label)

In [ ]:
vitals["feature_name"].isnull().sum()

np.int64(0)

In [ ]:
# convert to hourly time
vitals["charttime"] = pd.to_datetime(vitals["charttime"])
vitals["charttime_hour"] = vitals["charttime"].dt.floor("h")

In [ ]:
# aggregate per hour
vitals_hourly = (
    vitals.groupby(["stay_id", "charttime_hour", "feature_name"])["valuenum"]
    .mean()
    .reset_index()
)

In [ ]:
# pivot to wide format
vitals_wide = vitals_hourly.pivot_table(
    index=["stay_id", "charttime_hour"],
    columns="feature_name",
    values="valuenum"
).reset_index()

In [ ]:
print(vitals_wide.shape)
vitals_wide.head()

(7904172, 9)


feature_name,stay_id,charttime_hour,abp_dia,abp_mean,abp_sys,heart_rate,resp_rate,spo2,temp_c
0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,100.0,18.0,100.0,NaN
1,30000153,2174-09-29 13:00:00,72.0,NaN,151.0,104.0,16.0,100.0,NaN
2,30000153,2174-09-29 14:00:00,61.0,80.0,131.0,83.0,16.0,100.0,NaN
3,30000153,2174-09-29 15:00:00,65.0,84.0,123.0,92.0,14.0,100.0,NaN
4,30000153,2174-09-29 16:00:00,55.0,71.0,109.0,83.0,16.0,100.0,NaN


In [ ]:
vitals_wide.to_csv(output_dir / "vitals_hourly_wide.csv", index=False)
print("Hourly wide vitals saved successfully")

Hourly wide vitals saved successfully


In [ ]:
vitals_wide.columns

Index(['stay_id', 'charttime_hour', 'abp_dia', 'abp_mean', 'abp_sys',
       'heart_rate', 'resp_rate', 'spo2', 'temp_c'],
      dtype='object', name='feature_name')

In [ ]:
vitals_wide.columns.name = None
vitals_wide.columns

Index(['stay_id', 'charttime_hour', 'abp_dia', 'abp_mean', 'abp_sys',
       'heart_rate', 'resp_rate', 'spo2', 'temp_c'],
      dtype='object')

In [ ]:
# merge icu admission time
cohort_small = cohort[["stay_id", "intime"]].copy()
vitals_wide = vitals_wide.merge(cohort_small, on="stay_id", how="left")
print(vitals_wide.shape)
vitals_wide.head()

(7904172, 10)


,stay_id,charttime_hour,abp_dia,abp_mean,abp_sys,heart_rate,resp_rate,spo2,temp_c,intime
0,30000153,2174-09-29 12:00:00,NaN,NaN,NaN,100.0,18.0,100.0,NaN,2174-09-29 12:09:00
1,30000153,2174-09-29 13:00:00,72.0,NaN,151.0,104.0,16.0,100.0,NaN,2174-09-29 12:09:00
2,30000153,2174-09-29 14:00:00,61.0,80.0,131.0,83.0,16.0,100.0,NaN,2174-09-29 12:09:00
3,30000153,2174-09-29 15:00:00,65.0,84.0,123.0,92.0,14.0,100.0,NaN,2174-09-29 12:09:00
4,30000153,2174-09-29 16:00:00,55.0,71.0,109.0,83.0,16.0,100.0,NaN,2174-09-29 12:09:00


In [ ]:
# times dataframe
vitals_wide["charttime_hour"] = pd.to_datetime(vitals_wide["charttime_hour"])
vitals_wide["intime"] = pd.to_datetime(vitals_wide["intime"])

In [ ]:
# calculate hours since icu admission
vitals_wide["hours_since_admit"] = (
    vitals_wide["charttime_hour"] - vitals_wide["intime"]
).dt.total_seconds() / 3600

vitals_wide["hours_since_admit"].describe()

,hours_since_admit
count,7.904172e+06
mean,1.399220e+02
std,2.280014e+02
min,-8.760517e+03
25%,2.276222e+01
50%,6.191111e+01
75%,1.646381e+02
max,8.851600e+03


In [ ]:
# kepping only first 24 hours
vitals_24h = vitals_wide[
    (vitals_wide["hours_since_admit"] >= 0) &
    (vitals_wide["hours_since_admit"] < 24)
].copy()

print(vitals_24h.shape)
vitals_24h.head()

(2002030, 11)


,stay_id,charttime_hour,abp_dia,abp_mean,abp_sys,heart_rate,resp_rate,spo2,temp_c,intime,hours_since_admit
1,30000153,2174-09-29 13:00:00,72.0,NaN,151.0,104.0,16.0,100.0,NaN,2174-09-29 12:09:00,0.85
2,30000153,2174-09-29 14:00:00,61.0,80.0,131.0,83.0,16.0,100.0,NaN,2174-09-29 12:09:00,1.85
3,30000153,2174-09-29 15:00:00,65.0,84.0,123.0,92.0,14.0,100.0,NaN,2174-09-29 12:09:00,2.85
4,30000153,2174-09-29 16:00:00,55.0,71.0,109.0,83.0,16.0,100.0,NaN,2174-09-29 12:09:00,3.85
5,30000153,2174-09-29 17:00:00,56.0,71.0,111.0,103.0,20.0,100.0,NaN,2174-09-29 12:09:00,4.85


In [ ]:
# create integer hour bins
vitals_24h["hour"] = vitals_24h["hours_since_admit"].astype(int)
print(vitals_24h["hour"].min(), vitals_24h["hour"].max())

0 23


In [ ]:
# keep only needed columns
vitals_24h = vitals_24h[
    [
        "stay_id",
        "hour",
        "abp_dia",
        "abp_mean",
        "abp_sys",
        "heart_rate",
        "resp_rate",
        "spo2",
        "temp_c",
    ]
].copy()

vitals_24h.head()

,stay_id,hour,abp_dia,abp_mean,abp_sys,heart_rate,resp_rate,spo2,temp_c
1,30000153,0,72.0,NaN,151.0,104.0,16.0,100.0,NaN
2,30000153,1,61.0,80.0,131.0,83.0,16.0,100.0,NaN
3,30000153,2,65.0,84.0,123.0,92.0,14.0,100.0,NaN
4,30000153,3,55.0,71.0,109.0,83.0,16.0,100.0,NaN
5,30000153,4,56.0,71.0,111.0,103.0,20.0,100.0,NaN


In [ ]:
# checking aggregation again incase there are repeated hours
vitals_24h = (
    vitals_24h.groupby(["stay_id", "hour"], as_index=False)
    .mean(numeric_only=True)
)
print(vitals_24h.shape)
vitals_24h.head()

(2002030, 9)


,stay_id,hour,abp_dia,abp_mean,abp_sys,heart_rate,resp_rate,spo2,temp_c
0,30000153,0,72.0,NaN,151.0,104.0,16.0,100.0,NaN
1,30000153,1,61.0,80.0,131.0,83.0,16.0,100.0,NaN
2,30000153,2,65.0,84.0,123.0,92.0,14.0,100.0,NaN
3,30000153,3,55.0,71.0,109.0,83.0,16.0,100.0,NaN
4,30000153,4,56.0,71.0,111.0,103.0,20.0,100.0,NaN


# save new file vitals_24h

In [ ]:
vitals_24h.to_csv(output_dir / "vitals_24h.csv", index=False)
print("vitals_24h.csv saved successfully")

vitals_24h.csv saved successfully


In [ ]:
vitals_24h.groupby("stay_id")["hour"].nunique().describe()

,hour
count,93630.000000
mean,21.382356
std,4.216299
min,1.000000
25%,21.000000
50%,23.000000
75%,24.000000
max,24.000000


## Fixing patients stay

In [ ]:
# removing patients with less hours and only keeping >12
valid_stays = vitals_24h.groupby("stay_id")["hour"].nunique()
valid_stays = valid_stays[valid_stays >= 12].index
vitals_24h = vitals_24h[vitals_24h["stay_id"].isin(valid_stays)]

print(vitals_24h.shape)

(1967191, 9)


In [ ]:
# create full 24h grid, making sure each patient has 24 rows
import pandas as pd

all_hours = pd.DataFrame({"hour": list(range(24))})
vitals_complete = []
for stay_id, group in vitals_24h.groupby("stay_id"):
    group = group.set_index("hour")
    group = group.reindex(range(24))
    group["stay_id"] = stay_id
    vitals_complete.append(group.reset_index())
vitals_complete = pd.concat(vitals_complete, ignore_index=True)

print(vitals_complete.shape)

(2137800, 9)


In [ ]:
# FIXED: forward fill with limit=4 (max 4 hour carry-forward)
# then backward fill only for the very start of stay (limit=1)
vitals_complete = vitals_complete.sort_values(['stay_id', 'hour'])
vitals_complete = (
    vitals_complete
    .groupby('stay_id', group_keys=False)
    .apply(lambda x: x.ffill(limit=4).bfill(limit=1))
)
vitals_complete.reset_index(drop=True, inplace=True)

/tmp/ipykernel_6079/1126807257.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.ffill(limit=4).bfill(limit=1))


In [ ]:
vitals_complete.groupby("stay_id")["hour"].nunique().unique()

array([24])

In [ ]:
vitals_complete.to_csv(output_dir / "vitals_complete.csv", index=False)
print("Final LSTM-ready dataset saved")

Final LSTM-ready dataset saved


In [ ]:
vitals_complete.shape

(2137800, 9)

In [ ]:
# setting the feature column
feature_cols = [
    "abp_dia",
    "abp_mean",
    "abp_sys",
    "heart_rate",
    "resp_rate",
    "spo2",
    "temp_c"
]

In [ ]:
# sorting data properly
vitals_complete = vitals_complete.sort_values(["stay_id", "hour"]).reset_index(drop=True)

In [ ]:
# getting stay orders
stay_ids_order = vitals_complete["stay_id"].drop_duplicates().tolist()
print("Number of stays:", len(stay_ids_order))

Number of stays: 89075


In [ ]:
# building 3d array (number of stays, 24 , 7)
import numpy as np
X = np.array([
    vitals_complete[vitals_complete["stay_id"] == stay_id][feature_cols].values
    for stay_id in stay_ids_order
])

print("X shape:", X.shape)

X shape: (89075, 24, 7)


## Load labevents for SOFA

In [ ]:
import zipfile
lab_zip_path = DATA_DIR / "chartevents_labevents.zip"
with zipfile.ZipFile(lab_zip_path, "r") as z:
    print(z.namelist())

['Cleaned/', 'Cleaned/chartevents.csv', 'Cleaned/labevents.csv']


In [ ]:
# check whether d_labitems is available
for file in DATA_DIR.iterdir():
    print(file.name)

inputevents.csv
outputevents.csv
procedureevents.csv
ingredientevents.csv
datetimeevents.csv
icustays.csv
d_items.csv
patients.csv
prescriptions.csv
admissions.csv
microbiologyevents.csv
caregiver.csv
d_labitems.csv
chartevents_labevents.zip
icu_cohort.csv


In [ ]:
# loading d_labitems and inspect its columns
d_labitems = pd.read_csv(DATA_DIR / "d_labitems.csv")
print(d_labitems.shape)
d_labitems.head()

(1650, 4)


,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


In [ ]:
d_labitems.columns

Index(['itemid', 'label', 'fluid', 'category'], dtype='object')

In [ ]:
# search for three labs
# (Platelet)
d_labitems[d_labitems["label"].str.contains("platelet", case=False, na=False)]

,itemid,label,fluid,category
425,51240,Large Platelets,Blood,Hematology
449,51264,Platelet Clumps,Blood,Hematology
450,51265,Platelet Count,Blood,Hematology
451,51266,Platelet Smear,Blood,Hematology
1195,52105,Direct Antiplatelet Antibodies,Blood,Hematology
1231,52142,Mean Platelet Volume,Blood,Hematology
1248,52159,Platelet Aggregation,Blood,Hematology
1648,53189,Platelet Count,Blood,Chemistry


In [ ]:
# (bilirubin)
d_labitems[d_labitems["label"].str.contains("bilirubin", case=False, na=False)]

,itemid,label,fluid,category
36,50838,"Bilirubin, Total, Ascites",Ascites,Chemistry
81,50883,"Bilirubin, Direct",Blood,Chemistry
82,50884,"Bilirubin, Indirect",Blood,Chemistry
83,50885,"Bilirubin, Total",Blood,Chemistry
216,51028,"Bilirubin, Total, Body Fluid",Other Body Fluid,Chemistry
237,51049,"Bilirubin, Total, Pleural",Pleural,Chemistry
624,51464,Bilirubin,Urine,Hematology
625,51465,Bilirubin Crystals,Urine,Hematology
691,51568,"Bilirubin, Neonatal",Blood,Chemistry
692,51569,"Bilirubin, Neonatal, Direct",Blood,Chemistry


In [ ]:
# (creatinine)
d_labitems[d_labitems["label"].str.contains("creatinine", case=False, na=False)]

,itemid,label,fluid,category
39,50841,"Creatinine, Ascites",Ascites,Chemistry
110,50912,Creatinine,Blood,Chemistry
209,51021,"Creatinine, Joint Fluid",Joint Fluid,Chemistry
220,51032,"Creatinine, Body Fluid",Other Body Fluid,Chemistry
240,51052,"Creatinine, Pleural",Pleural,Chemistry
255,51067,24 hr Creatinine,Urine,Chemistry
258,51070,"Albumin/Creatinine, Urine",Urine,Chemistry
261,51073,"Amylase/Creatinine Ratio, Urine",Urine,Chemistry
268,51080,Creatinine Clearance,Urine,Chemistry
269,51081,"Creatinine, Serum",Urine,Chemistry


In [ ]:
# final itemids to use
# 51265 platelet count (blood, hematology)
# 50855 bilirubin, total (blood)
# 1560 bilirubin, total (blood)
# 52546 and 50912 creatinine (blood)

In [ ]:
SOFA_LAB_ITEMIDS = {
    "platelets": [51265],
    "bilirubin": [50885],
    "creatinine": [50912, 52546]
}

## New file sofa_labs_filtered

In [ ]:
# extract labs from labevents
import zipfile
import pandas as pd
lab_zip_path = DATA_DIR / "chartevents_labevents.zip"
output_file = output_dir / "sofa_labs_filtered.csv"
if output_file.exists():
    output_file.unlink()
chunk_size = 100000
first_chunk = True
all_itemids = (
    SOFA_LAB_ITEMIDS["platelets"] +
    SOFA_LAB_ITEMIDS["bilirubin"] +
    SOFA_LAB_ITEMIDS["creatinine"]
)

stay_ids = set(cohort["stay_id"].dropna().unique())
with zipfile.ZipFile(lab_zip_path, "r") as z:
    with z.open("Cleaned/labevents.csv") as f:
        for i, chunk in enumerate(
            pd.read_csv(
                f,
                chunksize=chunk_size,
                usecols=[
                    "subject_id",
                    "hadm_id",
                    "charttime",
                    "itemid",
                    "valuenum"
                ]
            )
        ):
            chunk = chunk[chunk["itemid"].isin(all_itemids)]

            # Keep only stays in our cohort
            chunk = chunk.merge(
                cohort[["subject_id", "hadm_id", "stay_id"]],
                on=["subject_id", "hadm_id"],
                how="inner"
            )

            if not chunk.empty:
                chunk.to_csv(output_file, mode="a", header=first_chunk, index=False)
                first_chunk = False

            print(f"Finished chunk {i+1}")
print("SOFA lab data saved")

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11
Finished chunk 12
Finished chunk 13
Finished chunk 14
Finished chunk 15
Finished chunk 16
Finished chunk 17
Finished chunk 18
Finished chunk 19
Finished chunk 20
Finished chunk 21
Finished chunk 22
Finished chunk 23
Finished chunk 24
Finished chunk 25
Finished chunk 26
Finished chunk 27
Finished chunk 28
Finished chunk 29
Finished chunk 30
Finished chunk 31
Finished chunk 32
Finished chunk 33
Finished chunk 34
Finished chunk 35
Finished chunk 36
Finished chunk 37
Finished chunk 38
Finished chunk 39
Finished chunk 40
Finished chunk 41
Finished chunk 42
Finished chunk 43
Finished chunk 44
Finished chunk 45
Finished chunk 46
Finished chunk 47
Finished chunk 48
Finished chunk 49
Finished chunk 50
Finished chunk 51
Finished chunk 52
Finished chunk 53
Finished chunk 54
Finished chunk 55
Finished chunk 56
F

In [ ]:
labs = pd.read_csv(output_file)
labs["charttime"] = pd.to_datetime(labs["charttime"])
print(labs.shape)
labs.head()

(3233165, 6)


,subject_id,hadm_id,itemid,charttime,valuenum,stay_id
0,10000032,29079034.0,50912,2180-07-23 21:45:00,0.5,39553978
1,10000032,29079034.0,50885,2180-07-24 06:35:00,2.7,39553978
2,10000032,29079034.0,50912,2180-07-24 06:35:00,0.4,39553978
3,10000032,29079034.0,51265,2180-07-24 06:35:00,94.0,39553978
4,10000032,29079034.0,50885,2180-07-25 04:45:00,1.7,39553978


In [ ]:
labs.shape

(3233165, 6)

In [ ]:
labs = labs.dropna(subset=["valuenum"])
print(labs.shape)
print(labs.head())

(3219991, 6)
   subject_id     hadm_id  itemid           charttime  valuenum   stay_id
0    10000032  29079034.0   50912 2180-07-23 21:45:00       0.5  39553978
1    10000032  29079034.0   50885 2180-07-24 06:35:00       2.7  39553978
2    10000032  29079034.0   50912 2180-07-24 06:35:00       0.4  39553978
3    10000032  29079034.0   51265 2180-07-24 06:35:00      94.0  39553978
4    10000032  29079034.0   50885 2180-07-25 04:45:00       1.7  39553978


In [ ]:
# map itemids to lab names
lab_itemid_to_name = {
    51265: "platelets",
    50885: "bilirubin",
    53089: "bilirubin",
    50912: "creatinine",
    52546: "creatinine"
}
labs["lab_name"] = labs["itemid"].map(lab_itemid_to_name)
print(labs["lab_name"].isnull().sum())
labs.head()

0


,subject_id,hadm_id,itemid,charttime,valuenum,stay_id,lab_name
0,10000032,29079034.0,50912,2180-07-23 21:45:00,0.5,39553978,creatinine
1,10000032,29079034.0,50885,2180-07-24 06:35:00,2.7,39553978,bilirubin
2,10000032,29079034.0,50912,2180-07-24 06:35:00,0.4,39553978,creatinine
3,10000032,29079034.0,51265,2180-07-24 06:35:00,94.0,39553978,platelets
4,10000032,29079034.0,50885,2180-07-25 04:45:00,1.7,39553978,bilirubin


In [ ]:
# convert to hourly bins
labs["charttime_hour"] = labs["charttime"].dt.floor("h")
labs.head()

,subject_id,hadm_id,itemid,charttime,valuenum,stay_id,lab_name,charttime_hour
0,10000032,29079034.0,50912,2180-07-23 21:45:00,0.5,39553978,creatinine,2180-07-23 21:00:00
1,10000032,29079034.0,50885,2180-07-24 06:35:00,2.7,39553978,bilirubin,2180-07-24 06:00:00
2,10000032,29079034.0,50912,2180-07-24 06:35:00,0.4,39553978,creatinine,2180-07-24 06:00:00
3,10000032,29079034.0,51265,2180-07-24 06:35:00,94.0,39553978,platelets,2180-07-24 06:00:00
4,10000032,29079034.0,50885,2180-07-25 04:45:00,1.7,39553978,bilirubin,2180-07-25 04:00:00


In [ ]:
# aggregate to each hour, take mean if multiple values
labs_hourly = (
    labs.groupby(["stay_id", "charttime_hour", "lab_name"])["valuenum"]
    .mean()
    .reset_index()
)

print(labs_hourly.shape)
labs_hourly.head()

(3214497, 4)


,stay_id,charttime_hour,lab_name,valuenum
0,30000153,2174-09-29 15:00:00,creatinine,0.9
1,30000153,2174-09-29 15:00:00,platelets,173.0
2,30000153,2174-09-30 03:00:00,creatinine,1.1
3,30000153,2174-09-30 03:00:00,platelets,162.0
4,30000153,2174-10-01 05:00:00,creatinine,0.8


In [ ]:
# convert to wide format
labs_wide = labs_hourly.pivot_table(
    index=["stay_id", "charttime_hour"],
    columns="lab_name",
    values="valuenum"
).reset_index()

labs_wide.columns.name = None

print(labs_wide.shape)
labs_wide.head()

(1623260, 5)


,stay_id,charttime_hour,bilirubin,creatinine,platelets
0,30000153,2174-09-29 15:00:00,NaN,0.9,173.0
1,30000153,2174-09-30 03:00:00,NaN,1.1,162.0
2,30000153,2174-10-01 05:00:00,NaN,0.8,112.0
3,30000153,2174-10-02 07:00:00,NaN,0.8,120.0
4,30000153,2174-10-03 10:00:00,NaN,0.8,152.0


In [ ]:
# saving to sofa labs
labs_wide.to_csv(output_dir / "sofa_labs_hourly_wide.csv", index=False)
print("sofa_labs_hourly_wide.csv saved successfully")

sofa_labs_hourly_wide.csv saved successfully


In [ ]:
# platelets score
def sofa_platelets(x):
    if pd.isna(x):
        return np.nan
    elif x >= 150:
        return 0
    elif x >= 100:
        return 1
    elif x >= 50:
        return 2
    elif x >= 20:
        return 3
    else:
        return 4

labs_wide["sofa_platelets"] = labs_wide["platelets"].apply(sofa_platelets)

In [ ]:
# bilirubin score
def sofa_bilirubin(x):
    if pd.isna(x):
        return np.nan
    elif x < 1.2:
        return 0
    elif x < 2.0:
        return 1
    elif x < 6.0:
        return 2
    elif x < 12.0:
        return 3
    else:
        return 4
labs_wide["sofa_bilirubin"] = labs_wide["bilirubin"].apply(sofa_bilirubin)

In [ ]:
# creatinine score
def sofa_creatinine(x):
    if pd.isna(x):
        return np.nan
    elif x < 1.2:
        return 0
    elif x < 2.0:
        return 1
    elif x < 3.5:
        return 2
    elif x < 5.0:
        return 3
    else:
        return 4
labs_wide["sofa_creatinine"] = labs_wide["creatinine"].apply(sofa_creatinine)

In [ ]:
labs_wide.head()

,stay_id,charttime_hour,bilirubin,creatinine,platelets,sofa_platelets,sofa_bilirubin,sofa_creatinine
0,30000153,2174-09-29 15:00:00,NaN,0.9,173.0,0.0,NaN,0.0
1,30000153,2174-09-30 03:00:00,NaN,1.1,162.0,0.0,NaN,0.0
2,30000153,2174-10-01 05:00:00,NaN,0.8,112.0,1.0,NaN,0.0
3,30000153,2174-10-02 07:00:00,NaN,0.8,120.0,1.0,NaN,0.0
4,30000153,2174-10-03 10:00:00,NaN,0.8,152.0,0.0,NaN,0.0


In [ ]:
labs_wide[[
    "sofa_platelets",
    "sofa_bilirubin",
    "sofa_creatinine"
]].describe()

,sofa_platelets,sofa_bilirubin,sofa_creatinine
count,1.343872e+06,430827.000000,1.439798e+06
mean,6.427800e-01,0.977945,7.795663e-01
std,1.010800e+00,1.332030,1.098276e+00
min,0.000000e+00,0.000000,0.000000e+00
25%,0.000000e+00,0.000000,0.000000e+00
50%,0.000000e+00,0.000000,0.000000e+00
75%,1.000000e+00,2.000000,1.000000e+00
max,4.000000e+00,4.000000,4.000000e+00


In [ ]:
labs_wide["sofa_lab_total"] = (
    labs_wide["sofa_platelets"] +
    labs_wide["sofa_bilirubin"] +
    labs_wide["sofa_creatinine"]
)
labs_wide["sofa_lab_total"].describe()

,sofa_lab_total
count,404554.000000
mean,2.828164
std,2.449542
min,0.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,12.000000


In [ ]:
# FIXED: Full cardiovascular SOFA including vasopressors
# Step 1: MAP component (score 0 or 1)
map_df = vitals_wide[['stay_id', 'charttime_hour', 'abp_mean']].copy()

def sofa_map(x):
    if pd.isna(x): return np.nan
    return 0 if x >= 70 else 1

map_df['sofa_cardio'] = map_df['abp_mean'].apply(sofa_map)

# Step 2: Extract vasopressor data from inputevents
# Vasopressors push score to 2, 3, or 4
VASOPRESSOR_ITEMIDS = {
    'dopamine':       221662,
    'dobutamine':     221653,
    'norepinephrine': 221906,
    'epinephrine':    221289,
    'phenylephrine':  221749,
    'vasopressin':    222315,
}

vaso_output = OUTPUT_DIR / 'vasopressors_filtered.csv'
if vaso_output.exists():
    vaso_output.unlink()

stay_ids = set(cohort['stay_id'].dropna().unique())
all_vaso_ids = list(VASOPRESSOR_ITEMIDS.values())
first_chunk = True

# inputevents is NOT in the zip — it's a separate CSV in DATA_DIR
with open(DATA_DIR / 'inputevents.csv') as f:
    for i, chunk in enumerate(pd.read_csv(
        f, chunksize=100_000,
        usecols=['stay_id', 'starttime', 'itemid', 'amount', 'rate', 'rateuom']
    )):
        chunk = chunk[
            chunk['stay_id'].isin(stay_ids) &
            chunk['itemid'].isin(all_vaso_ids)
        ]
        if not chunk.empty:
            chunk.to_csv(vaso_output, mode='a', header=first_chunk, index=False)
            first_chunk = False
        print(f'chunk {i+1}', end='\r')

print('\nVasopressor data saved')

chunk 110
Vasopressor data saved


In [ ]:
vaso_df = pd.read_csv(vaso_output)
vaso_df['starttime'] = pd.to_datetime(vaso_df['starttime'])
vaso_df['charttime_hour'] = vaso_df['starttime'].dt.floor('h')

# Flag any vasopressor use per stay-hour (score ≥ 2)
# Exact dose-based scoring (score 3 and 4) requires weight-adjusted rates
# which are unreliable in MIMIC — binary vasopressor flag giving score=2
# is the approach used in most published MIMIC-IV SOFA implementations
vaso_flag = (
    vaso_df.groupby(['stay_id', 'charttime_hour'])
    .size()
    .reset_index(name='vaso_count')
)
vaso_flag['sofa_cardio_vaso'] = 2  # any vasopressor use = score 2

# Merge into map_df and take the max of MAP score and vasopressor score
map_df = map_df.merge(
    vaso_flag[['stay_id', 'charttime_hour', 'sofa_cardio_vaso']],
    on=['stay_id', 'charttime_hour'],
    how='left'
)
map_df['sofa_cardio_vaso'] = map_df['sofa_cardio_vaso'].fillna(0)
map_df['sofa_cardio'] = map_df[['sofa_cardio', 'sofa_cardio_vaso']].max(axis=1)

print(map_df['sofa_cardio'].value_counts())

sofa_cardio
0.0    6796240
1.0     599703
2.0     508229
Name: count, dtype: int64


In [ ]:
d_items[d_items['label'].str.contains('GCS', case=False, na=False)]

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
159,220739,GCS - Eye Opening,Eye Opening,chartevents,Neurological,NaN,Text,NaN,NaN
407,223900,GCS - Verbal Response,Verbal Response,chartevents,Neurological,NaN,Text,NaN,NaN
408,223901,GCS - Motor Response,Motor Response,chartevents,Neurological,NaN,Text,NaN,NaN
1965,226755,GcsApacheIIScore,GcsApacheIIScore,chartevents,Scores - APACHE II,NaN,Numeric,NaN,NaN
1966,226756,GCSEyeApacheIIValue,GCSEyeApacheIIValue,chartevents,Scores - APACHE II,NaN,Text,NaN,NaN
1967,226757,GCSMotorApacheIIValue,GCSMotorApacheIIValue,chartevents,Scores - APACHE II,NaN,Text,NaN,NaN
1968,226758,GCSVerbalApacheIIValue,GCSVerbalApacheIIValue,chartevents,Scores - APACHE II,NaN,Text,NaN,NaN
2054,227011,GCSEye_ApacheIV,GCSEye_ApacheIV,chartevents,Scores - APACHE IV (2),NaN,Text,NaN,NaN
2055,227012,GCSMotor_ApacheIV,GCSMotor_ApacheIV,chartevents,Scores - APACHE IV (2),NaN,Text,NaN,NaN
2056,227013,GcsScore_ApacheIV,GcsScore_ApacheIV,chartevents,Scores - APACHE IV (2),NaN,Numeric,NaN,NaN


In [ ]:
import zipfile
import pandas as pd

chartevents_zip_path = DATA_DIR / "chartevents_labevents.zip"
output_file = output_dir / "gcs_filtered.csv"

if output_file.exists():
    output_file.unlink()

GCS_ITEMIDS = [220739, 223900, 223901]

chunk_size = 100000
first_chunk = True

stay_ids = set(cohort["stay_id"].dropna().unique())

with zipfile.ZipFile(chartevents_zip_path, "r") as z:
    with z.open("Cleaned/chartevents.csv") as f:
        for i, chunk in enumerate(
            pd.read_csv(
                f,
                chunksize=chunk_size,
                usecols=[
                    "subject_id",
                    "hadm_id",
                    "stay_id",
                    "charttime",
                    "itemid",
                    "valuenum"
                ]
            )
        ):
            chunk = chunk[
                (chunk["stay_id"].isin(stay_ids)) &
                (chunk["itemid"].isin(GCS_ITEMIDS))
            ]

            if not chunk.empty:
                chunk.to_csv(output_file, mode="a", header=first_chunk, index=False)
                first_chunk = False

            print(f"Finished chunk {i+1}")

print("GCS data saved")

Finished chunk 1
Finished chunk 2
Finished chunk 3
Finished chunk 4
Finished chunk 5
Finished chunk 6
Finished chunk 7
Finished chunk 8
Finished chunk 9
Finished chunk 10
Finished chunk 11
Finished chunk 12
Finished chunk 13
Finished chunk 14
Finished chunk 15
Finished chunk 16
Finished chunk 17
Finished chunk 18
Finished chunk 19
Finished chunk 20
Finished chunk 21
Finished chunk 22
Finished chunk 23
Finished chunk 24
Finished chunk 25
Finished chunk 26
Finished chunk 27
Finished chunk 28
Finished chunk 29
Finished chunk 30
Finished chunk 31
Finished chunk 32
Finished chunk 33
Finished chunk 34
Finished chunk 35
Finished chunk 36
Finished chunk 37
Finished chunk 38
Finished chunk 39
Finished chunk 40
Finished chunk 41
Finished chunk 42
Finished chunk 43
Finished chunk 44
Finished chunk 45
Finished chunk 46
Finished chunk 47
Finished chunk 48
Finished chunk 49
Finished chunk 50
Finished chunk 51
Finished chunk 52
Finished chunk 53
Finished chunk 54
Finished chunk 55
Finished chunk 56
F

In [ ]:
gcs = pd.read_csv(output_file)
gcs["charttime"] = pd.to_datetime(gcs["charttime"])
gcs = gcs.dropna(subset=["valuenum"])
print(gcs.shape)
gcs.head()

(6610876, 6)


,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
0,10000032,29079034,39553978,2180-07-23 14:45:00,220739,4.0
1,10000032,29079034,39553978,2180-07-23 14:45:00,223901,6.0
2,10000032,29079034,39553978,2180-07-23 14:45:00,223900,4.0
3,10000032,29079034,39553978,2180-07-23 18:22:00,220739,4.0
4,10000032,29079034,39553978,2180-07-23 18:22:00,223900,5.0


In [ ]:
gcs["charttime_hour"] = gcs["charttime"].dt.floor("h")

In [ ]:
gcs_map = {
    220739: "eye",
    223900: "verbal",
    223901: "motor"
}

gcs["component"] = gcs["itemid"].map(gcs_map)

In [ ]:
gcs['component'].isnull().sum()

np.int64(0)

In [ ]:
gcs["charttime_hour"] = gcs["charttime"].dt.floor("h")

gcs_hourly = (
    gcs.groupby(["stay_id", "charttime_hour", "component"])["valuenum"]
    .mean()
    .reset_index()
)

In [ ]:
gcs_wide = gcs_hourly.pivot_table(
    index=["stay_id", "charttime_hour"],
    columns="component",
    values="valuenum"
).reset_index()

gcs_wide.columns.name = None

print(gcs_wide.shape)
gcs_wide.head()

(2201532, 5)


,stay_id,charttime_hour,eye,motor,verbal
0,30000153,2174-09-29 12:00:00,3.0,5.0,1.0
1,30000153,2174-09-29 16:00:00,4.0,6.0,1.0
2,30000153,2174-09-29 17:00:00,4.0,6.0,1.0
3,30000153,2174-09-29 18:00:00,3.0,5.0,1.0
4,30000153,2174-09-29 19:00:00,3.0,5.0,1.0


In [ ]:
gcs_wide.shape

(2201532, 5)

In [ ]:
# compute total GCS
gcs_wide["gcs_total"] = gcs_wide[["eye", "verbal", "motor"]].sum(axis=1, min_count=3)
gcs_wide.head()

,stay_id,charttime_hour,eye,motor,verbal,gcs_total
0,30000153,2174-09-29 12:00:00,3.0,5.0,1.0,9.0
1,30000153,2174-09-29 16:00:00,4.0,6.0,1.0,11.0
2,30000153,2174-09-29 17:00:00,4.0,6.0,1.0,11.0
3,30000153,2174-09-29 18:00:00,3.0,5.0,1.0,9.0
4,30000153,2174-09-29 19:00:00,3.0,5.0,1.0,9.0


In [ ]:
# convert GCS to SOFA CNS score
def sofa_gcs(x):
    if pd.isna(x):
        return np.nan
    elif x == 15:
        return 0
    elif x >= 13:
        return 1
    elif x >= 10:
        return 2
    elif x >= 6:
        return 3
    else:
        return 4

gcs_wide["sofa_gcs"] = gcs_wide["gcs_total"].apply(sofa_gcs)
gcs_wide[["gcs_total", "sofa_gcs"]].describe()

,gcs_total,sofa_gcs
count,2.178092e+06,2.178092e+06
mean,1.170125e+01,1.397276e+00
std,3.809865e+00,1.376383e+00
min,3.000000e+00,0.000000e+00
25%,9.000000e+00,0.000000e+00
50%,1.400000e+01,1.000000e+00
75%,1.500000e+01,3.000000e+00
max,1.500000e+01,4.000000e+00


In [ ]:
# CLEAN Block 7 — single merge of all SOFA components
# At this point you have:
# labs_wide       → sofa_platelets, sofa_bilirubin, sofa_creatinine
# map_df          → sofa_cardio (MAP + vasopressors, from Block 6)
# gcs_wide        → sofa_gcs (from cells 123–124 above)

sofa_df = labs_wide.merge(
    map_df[['stay_id', 'charttime_hour', 'sofa_cardio']],
    on=['stay_id', 'charttime_hour'],
    how='outer'
)
sofa_df = sofa_df.merge(
    gcs_wide[['stay_id', 'charttime_hour', 'sofa_gcs']],
    on=['stay_id', 'charttime_hour'],
    how='outer'
)

# sofa_resp_final will be merged after Block 8 (respiratory component)
# For now verify the structure looks right
print(sofa_df.columns.tolist())
print(sofa_df.shape)

['stay_id', 'charttime_hour', 'bilirubin', 'creatinine', 'platelets', 'sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine', 'sofa_lab_total', 'sofa_cardio', 'sofa_gcs']
(8905908, 11)


In [ ]:
print(sofa_df.columns.tolist())

['stay_id', 'charttime_hour', 'bilirubin', 'creatinine', 'platelets', 'sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine', 'sofa_lab_total', 'sofa_cardio', 'sofa_gcs']


In [ ]:
# After Block 7 — do NOT compute sofa_total here yet
# Just verify the 5 components are present
print("Columns after Block 7:", sofa_df.columns.tolist())
print("Shape:", sofa_df.shape)

Columns after Block 7: ['stay_id', 'charttime_hour', 'bilirubin', 'creatinine', 'platelets', 'sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine', 'sofa_lab_total', 'sofa_cardio', 'sofa_gcs']
Shape: (8905908, 11)


## Respiratory Sofa

In [ ]:
# NEW: Respiratory SOFA component (PaO2/FiO2 ratio)
# SpO2/FiO2 (SF ratio) is used as a surrogate when PaO2 is unavailable
# SF ratio < 315 ≈ PF ratio < 400 (mild), per Sinha et al. 2020

# FiO2 from oxygen flow settings in chartevents
FIO2_ITEMIDS = [223835]  # FiO2 set (ventilated patients)
PAO2_ITEMIDS = [490, 779, 220224]  # PaO2 from ABG

resp_output = OUTPUT_DIR / 'resp_sofa_filtered.csv'
if resp_output.exists():
    resp_output.unlink()

RESP_ITEMIDS = FIO2_ITEMIDS + PAO2_ITEMIDS
first_chunk = True

with zipfile.ZipFile(chartevents_zip_path, 'r') as z:
    with z.open('Cleaned/chartevents.csv') as f:
        for i, chunk in enumerate(pd.read_csv(
            f, chunksize=100_000,
            usecols=['stay_id', 'charttime', 'itemid', 'valuenum']
        )):
            chunk = chunk[
                chunk['stay_id'].isin(stay_ids) &
                chunk['itemid'].isin(RESP_ITEMIDS)
            ]
            if not chunk.empty:
                chunk.to_csv(resp_output, mode='a', header=first_chunk, index=False)
                first_chunk = False
        print(f'chunk {i+1}', end='\r')

print('\nRespiratory data saved')

chunk 4330
Respiratory data saved


In [ ]:
resp_df = pd.read_csv(resp_output)
resp_df['charttime'] = pd.to_datetime(resp_df['charttime'])
resp_df['charttime_hour'] = resp_df['charttime'].dt.floor('h')

# Separate FiO2 and PaO2
fio2_df = resp_df[resp_df['itemid'].isin(FIO2_ITEMIDS)].copy()
pao2_df = resp_df[resp_df['itemid'].isin(PAO2_ITEMIDS)].copy()

fio2_hourly = fio2_df.groupby(['stay_id','charttime_hour'])['valuenum'].mean().reset_index()
fio2_hourly.columns = ['stay_id','charttime_hour','fio2']

pao2_hourly = pao2_df.groupby(['stay_id','charttime_hour'])['valuenum'].mean().reset_index()
pao2_hourly.columns = ['stay_id','charttime_hour','pao2']

pf_df = pao2_hourly.merge(fio2_hourly, on=['stay_id','charttime_hour'], how='left')

# FiO2 is a fraction (0–1) in MIMIC — convert if stored as percentage
pf_df.loc[pf_df['fio2'] > 1, 'fio2'] = pf_df.loc[pf_df['fio2'] > 1, 'fio2'] / 100

# PF ratio
pf_df['pf_ratio'] = pf_df['pao2'] / pf_df['fio2']

# For patients without ABG, use SF ratio surrogate
# SF ratio = SpO2 / FiO2 (using vitals_wide spo2)
sf_df = vitals_wide[['stay_id','charttime_hour','spo2']].merge(
    fio2_hourly, on=['stay_id','charttime_hour'], how='left'
)
sf_df['fio2'] = sf_df['fio2'].fillna(0.21)  # room air assumption
sf_df['sf_ratio'] = sf_df['spo2'] / sf_df['fio2']

def sofa_resp_pf(pf):
    if pd.isna(pf):   return np.nan
    elif pf >= 400:   return 0
    elif pf >= 300:   return 1
    elif pf >= 200:   return 2
    elif pf >= 100:   return 3
    else:             return 4

def sofa_resp_sf(sf):
    # SF surrogate thresholds per Rice et al. 2007
    if pd.isna(sf):   return np.nan
    elif sf >= 315:   return 0
    elif sf >= 235:   return 1
    elif sf >= 150:   return 2
    else:             return 3  # SF ratio can't reliably distinguish score 3 vs 4

pf_df['sofa_resp'] = pf_df['pf_ratio'].apply(sofa_resp_pf)
sf_df['sofa_resp_sf'] = sf_df['sf_ratio'].apply(sofa_resp_sf)

# Merge respiratory SOFA into sofa_df
# Prefer PF-based score, fall back to SF-based
sofa_df = sofa_df.merge(
    pf_df[['stay_id','charttime_hour','sofa_resp']],
    on=['stay_id','charttime_hour'], how='left'
)
sofa_df = sofa_df.merge(
    sf_df[['stay_id','charttime_hour','sofa_resp_sf']],
    on=['stay_id','charttime_hour'], how='left'
)

# Use PF when available, otherwise SF surrogate
sofa_df['sofa_resp_final'] = sofa_df['sofa_resp'].combine_first(sofa_df['sofa_resp_sf'])

# Recompute full 6-component SOFA
sofa_components = ['sofa_platelets', 'sofa_bilirubin', 'sofa_creatinine',
                   'sofa_cardio', 'sofa_gcs', 'sofa_resp_final']
sofa_df['sofa_total'] = sofa_df[sofa_components].sum(axis=1, min_count=1)

print("Full 6-component SOFA computed")
print(sofa_df['sofa_total'].describe())

Full 6-component SOFA computed
count    8.905670e+06
mean     1.150109e+00
std      1.856560e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      2.000000e+01
Name: sofa_total, dtype: float64


In [ ]:
print("Full 6-component SOFA computed")
print(f"Shape: {sofa_df.shape}")
print(sofa_df['sofa_total'].describe())

Full 6-component SOFA computed
Shape: (8905908, 15)
count    8.905670e+06
mean     1.150109e+00
std      1.856560e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      2.000000e+01
Name: sofa_total, dtype: float64


## Building Suspected Infections TimeStamps

### sepsis = suspected infection + SOFA>= 2

In [ ]:
prescriptions = pd.read_csv(DATA_DIR / "prescriptions.csv")
micro = pd.read_csv(DATA_DIR / "microbiologyevents.csv")
print(prescriptions.shape)
print(micro.shape)

/tmp/ipykernel_6079/928474306.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  prescriptions = pd.read_csv(DATA_DIR / "prescriptions.csv")
/tmp/ipykernel_6079/928474306.py:2: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  micro = pd.read_csv(DATA_DIR / "microbiologyevents.csv")


(20292611, 21)
(3988224, 25)


In [ ]:
# inspecting columns
print(prescriptions.columns.tolist())
print(micro.columns.tolist())

['subject_id', 'hadm_id', 'pharmacy_id', 'poe_id', 'poe_seq', 'order_provider_id', 'starttime', 'stoptime', 'drug_type', 'drug', 'formulary_drug_cd', 'gsn', 'ndc', 'prod_strength', 'form_rx', 'dose_val_rx', 'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'doses_per_24_hrs', 'route']
['microevent_id', 'subject_id', 'hadm_id', 'micro_specimen_id', 'order_provider_id', 'chartdate', 'charttime', 'spec_itemid', 'spec_type_desc', 'test_seq', 'storedate', 'storetime', 'test_itemid', 'test_name', 'org_itemid', 'org_name', 'isolate_num', 'quantity', 'ab_itemid', 'ab_name', 'dilution_text', 'dilution_comparison', 'dilution_value', 'interpretation', 'comments']


In [ ]:
# build suspected_infection_time per stay using antibiotics (prescriptions), culture (microbiology) and timing rules (24h/72h)

In [ ]:
# clean the time columns
prescriptions["starttime"] = pd.to_datetime(prescriptions["starttime"], errors="coerce")
micro["charttime"] = pd.to_datetime(micro["charttime"], errors="coerce")
micro["chartdate"] = pd.to_datetime(micro["chartdate"], errors="coerce")
# if charttime is missing, fall back to chartdate
micro["culture_time"] = micro["charttime"]
micro.loc[micro["culture_time"].isna(), "culture_time"] = micro.loc[micro["culture_time"].isna(), "chartdate"]

In [ ]:
# keep only rows linked to my cohort
cohort_keys = cohort[["subject_id", "hadm_id", "stay_id"]].dropna().drop_duplicates()
presc = prescriptions.merge(
    cohort_keys,
    on=["subject_id", "hadm_id"],
    how="inner"
)
cultures = micro.merge(
    cohort_keys,
    on=["subject_id", "hadm_id"],
    how="inner"
)

In [ ]:
# keeping only useful columns
presc = presc[["subject_id", "hadm_id", "stay_id", "starttime", "drug"]].dropna(subset=["starttime"])
cultures = cultures[["subject_id", "hadm_id", "stay_id", "culture_time", "spec_type_desc"]].dropna(subset=["culture_time"])
print(presc.shape)
print(cultures.shape)

(10647569, 5)
(1119342, 5)


In [ ]:
# dont use all drugs in prescriptions.
presc["drug"].dropna().astype(str).str.lower().value_counts().head(50)

,count
drug,
0.9% sodium chloride,561900
insulin,443096
potassium chloride,442385
furosemide,319705
sodium chloride 0.9% flush,297837
5% dextrose,294114
bag,281044
magnesium sulfate,258127
metoprolol tartrate,221258


In [ ]:
antibiotic_keywords = [
    "cillin", "cef", "cefepime", "ceftriaxone", "ceftazidime",
    "vancomycin", "meropenem", "imipenem", "ertapenem",
    "piperacillin", "tazobactam", "zosyn",
    "ciprofloxacin", "levofloxacin", "moxifloxacin",
    "azithromycin", "clarithromycin",
    "metronidazole", "clindamycin",
    "gentamicin", "amikacin", "tobramycin",
    "doxycycline", "tigecycline",
    "trimethoprim", "sulfamethoxazole",
    "linezolid", "daptomycin"
]
pattern = "|".join(antibiotic_keywords)
abx = presc[presc["drug"].astype(str).str.lower().str.contains(pattern, na=False)].copy()
print(abx.shape)
abx.head()

(534365, 5)


,subject_id,hadm_id,stay_id,starttime,drug
26,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin
35,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin
48,10000690,25860671,37081114,2150-11-03 08:00:00,Vancomycin
54,10000690,25860671,37081114,2150-11-06 08:00:00,Vancomycin
69,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin


In [ ]:
# build antibiotic/culture pairs inside each admission
abx = abx.rename(columns={"starttime": "antibiotic_time"})
cultures = cultures.rename(columns={"culture_time": "culture_time"})
pairs = abx.merge(
    cultures,
    on=["subject_id", "hadm_id", "stay_id"],
    how="inner"
)
print(pairs.shape)
pairs.head()

(20489022, 7)


,subject_id,hadm_id,stay_id,antibiotic_time,drug,culture_time,spec_type_desc
0,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin,2150-11-03 01:25:00,URINE
1,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin,2150-11-05 17:38:00,BLOOD CULTURE
2,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin,2150-11-05 17:38:00,BLOOD CULTURE
3,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin,2150-11-05 17:38:00,URINE
4,10000690,25860671,37081114,2150-11-09 08:00:00,Vancomycin,2150-11-09 09:49:00,STOOL


In [ ]:
# computing time difference
# positive value = antibiotic after culture
# negative value = antibiotic before culture
pairs["time_diff_hours"] = (
    pairs["antibiotic_time"] - pairs["culture_time"]
).dt.total_seconds() / 3600.0

In [ ]:
# apply the suspected infection rule
suspected_pairs = pairs[
    (
        (pairs["time_diff_hours"] >= 0) &
        (pairs["time_diff_hours"] <= 72)
    ) |
    (
        (pairs["time_diff_hours"] < 0) &
        (pairs["time_diff_hours"] >= -24)
    )
].copy()
print(suspected_pairs.shape)

(4156155, 8)


In [ ]:
# define suspected infection timestamp
# culture time if it came first or antibiotic time if it came first
suspected_pairs["suspected_infection_time"] = suspected_pairs["culture_time"]
abx_first_mask = suspected_pairs["time_diff_hours"] < 0
suspected_pairs.loc[abx_first_mask, "suspected_infection_time"] = suspected_pairs.loc[abx_first_mask, "antibiotic_time"]

In [ ]:
# keep only earliest suspected infection per time stay
suspected_infection = (
    suspected_pairs.groupby("stay_id", as_index=False)["suspected_infection_time"]
    .min()
)

print(suspected_infection.shape)
suspected_infection.head()

(58999, 2)


,stay_id,suspected_infection_time
0,30000153,2174-10-03 21:53:00
1,30000484,2136-01-14 18:10:00
2,30000646,2194-04-29 01:00:00
3,30000831,2140-04-18 05:11:00
4,30001148,2156-08-29 13:11:00


In [ ]:
# save it
suspected_infection.to_csv(output_dir / "suspected_infection.csv", index=False)
print("suspected_infection.csv saved successfully")

suspected_infection.csv saved successfully


In [ ]:
# What we have now is:
# one row per stay_id , earliest suspected_infection_time

In [ ]:
print(suspected_infection.shape)
suspected_infection.head()
# we have 58999 stay with suspected infection

(58999, 2)


,stay_id,suspected_infection_time
0,30000153,2174-10-03 21:53:00
1,30000484,2136-01-14 18:10:00
2,30000646,2194-04-29 01:00:00
3,30000831,2140-04-18 05:11:00
4,30001148,2156-08-29 13:11:00


## Combining together

In [ ]:
# prepare the two tables
suspected_infection["suspected_infection_time"] = pd.to_datetime(
    suspected_infection["suspected_infection_time"], errors="coerce"
)

sofa_df["charttime_hour"] = pd.to_datetime(
    sofa_df["charttime_hour"], errors="coerce"
)

cohort["intime"] = pd.to_datetime(cohort["intime"], errors="coerce")

In [ ]:
# Restrict SOFA to ICU stay hours only
# sofa_df charttime_hour must fall between intime and outtime of the ICU stay
sofa_df = sofa_df.merge(
    cohort[['stay_id', 'intime', 'outtime']],
    on='stay_id',
    how='left'
)
sofa_df['intime']  = pd.to_datetime(sofa_df['intime'])
sofa_df['outtime'] = pd.to_datetime(sofa_df['outtime'])

sofa_df = sofa_df[
    (sofa_df['charttime_hour'] >= sofa_df['intime']) &
    (sofa_df['charttime_hour'] <  sofa_df['outtime'])
].copy()

print(f"SOFA rows after ICU restriction: {sofa_df.shape}")
print(f"SOFA total describe after restriction:")
print(sofa_df['sofa_total'].describe())

SOFA rows after ICU restriction: (7872636, 17)
SOFA total describe after restriction:
count    7.872420e+06
mean     1.108052e+00
std      1.849118e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.000000e+00
max      2.000000e+01
Name: sofa_total, dtype: float64


In [ ]:
# Restrict suspected infection time to ICU stay window
suspected_infection = suspected_infection.merge(
    cohort[['stay_id', 'intime', 'outtime']],
    on='stay_id',
    how='left'
)
suspected_infection['intime']  = pd.to_datetime(suspected_infection['intime'])
suspected_infection['outtime'] = pd.to_datetime(suspected_infection['outtime'])
suspected_infection['suspected_infection_time'] = pd.to_datetime(
    suspected_infection['suspected_infection_time']
)

# If infection was flagged before ICU admission, use intime as the reference
suspected_infection['suspected_infection_time'] = suspected_infection[
    ['suspected_infection_time', 'intime']
].max(axis=1)

# Drop if suspected infection is after ICU discharge
suspected_infection = suspected_infection[
    suspected_infection['suspected_infection_time'] < suspected_infection['outtime']
].copy()

suspected_infection = suspected_infection[['stay_id', 'suspected_infection_time']]
print(f"Suspected infection stays after ICU restriction: {len(suspected_infection):,}")

Suspected infection stays after ICU restriction: 55,391


In [ ]:
# Re-run sepsis onset build with ICU-restricted data
sofa_pos = sofa_df[sofa_df['sofa_total'] >= 2][['stay_id', 'charttime_hour']].copy()

sepsis_candidates = sofa_pos.merge(
    suspected_infection[['stay_id', 'suspected_infection_time']],
    on='stay_id',
    how='inner'
)
sepsis_candidates['suspected_infection_time'] = pd.to_datetime(
    sepsis_candidates['suspected_infection_time']
)

sepsis_candidates['diff_from_infection_hours'] = (
    sepsis_candidates['charttime_hour'] - sepsis_candidates['suspected_infection_time']
).dt.total_seconds() / 3600

sepsis_candidates = sepsis_candidates[
    (sepsis_candidates['diff_from_infection_hours'] >= -48) &
    (sepsis_candidates['diff_from_infection_hours'] <=  24)
].copy()

print(f'Sepsis candidates after fix: {len(sepsis_candidates):,}')

sepsis_onset = (
    sepsis_candidates
    .groupby('stay_id', as_index=False)['charttime_hour']
    .min()
    .rename(columns={'charttime_hour': 'sepsis_onset_time'})
)

print(f'Stays with sepsis onset: {len(sepsis_onset):,}')
print(sepsis_onset.head())

Sepsis candidates after fix: 456,406
Stays with sepsis onset: 45,823
    stay_id   sepsis_onset_time
0  30000484 2136-01-14 18:00:00
1  30000646 2194-04-29 06:00:00
2  30000831 2140-04-17 22:00:00
3  30001148 2156-08-30 14:00:00
4  30001396 2147-10-19 04:00:00


In [ ]:
label_df = cohort[['stay_id', 'intime']].merge(
    sepsis_onset, on='stay_id', how='left'
)
label_df['onset_hour'] = (
    label_df['sepsis_onset_time'] - label_df['intime']
).dt.total_seconds() / 3600

print("Onset hour distribution after fix:")
print(label_df['onset_hour'].dropna().describe())
print("\nOnset before hour 0 (should be 0 or near 0):",
      (label_df['onset_hour'] < 0).sum())
print("Onset in first 24h:",
      ((label_df['onset_hour'] >= 0) & (label_df['onset_hour'] < 24)).sum())
print("Onset in hours 24-48:",
      ((label_df['onset_hour'] >= 24) & (label_df['onset_hour'] < 48)).sum())
print("Onset after hour 48:",
      (label_df['onset_hour'] >= 48).sum())
print("No sepsis:", label_df['onset_hour'].isna().sum())

Onset hour distribution after fix:
count    45823.000000
mean         5.330140
std         18.946517
min          0.000000
25%          0.554306
50%          1.548333
75%          3.858472
max        766.364722
Name: onset_hour, dtype: float64

Onset before hour 0 (should be 0 or near 0): 0
Onset in first 24h: 44341
Onset in hours 24-48: 637
Onset after hour 48: 845
No sepsis: 47907


In [ ]:
# ============================================================
# ROLLING HOURLY LABELS — Option B
# For each patient-hour T, label = 1 if sepsis onset in (T, T+6]
# ============================================================

PREDICTION_HORIZON_HOURS = 6

# Step 1: Build full hourly grid per stay
cohort['intime']  = pd.to_datetime(cohort['intime'])
cohort['outtime'] = pd.to_datetime(cohort['outtime'])

hourly_rows = []
for _, row in cohort.iterrows():
    n_hours = max(int(row['icu_los_hours']), 1)
    for t in range(n_hours):
        hourly_rows.append({
            'stay_id':    row['stay_id'],
            'subject_id': row['subject_id'],
            'hour':       t,
            'abs_time':   row['intime'] + pd.Timedelta(hours=t)
        })

hourly_grid = pd.DataFrame(hourly_rows)
print(f'Hourly grid shape: {hourly_grid.shape}')

# Step 2: Merge sepsis onset time
hourly_grid = hourly_grid.merge(
    sepsis_onset[['stay_id', 'sepsis_onset_time']],
    on='stay_id', how='left'
)
hourly_grid['sepsis_onset_time'] = pd.to_datetime(hourly_grid['sepsis_onset_time'])

# Step 3: Compute hours until sepsis onset
hourly_grid['hours_to_onset'] = (
    hourly_grid['sepsis_onset_time'] - hourly_grid['abs_time']
).dt.total_seconds() / 3600

# Step 4: Assign labels
hourly_grid['label'] = 0

# Positive: onset within prediction horizon
positive_mask = (
    (hourly_grid['hours_to_onset'] > 0) &
    (hourly_grid['hours_to_onset'] <= PREDICTION_HORIZON_HOURS)
)
hourly_grid.loc[positive_mask, 'label'] = 1

# Exclude: patient already septic at time T
already_septic_mask = hourly_grid['hours_to_onset'] <= 0
hourly_grid.loc[already_septic_mask, 'label'] = np.nan

# Step 5: Require minimum 1 hour of history
# Hour 0 = admission hour, model has no prior data to learn from
hourly_grid = hourly_grid[hourly_grid['hour'] >= 1].copy()

# Step 6: Drop already-septic rows
hourly_grid = hourly_grid.dropna(subset=['label'])
hourly_grid['label'] = hourly_grid['label'].astype(int)

# Step 7: Detailed summary
print(f'\nAfter filtering:')
print(f'Total prediction points : {len(hourly_grid):,}')
print(f'Positive labels         : {hourly_grid["label"].sum():,}')
print(f'Negative labels         : {(hourly_grid["label"]==0).sum():,}')
print(f'Positive rate           : {hourly_grid["label"].mean():.3%}')
print(f'Unique stays            : {hourly_grid["stay_id"].nunique():,}')
print(f'Unique patients         : {hourly_grid["subject_id"].nunique():,}')

# Step 8: Check how many sepsis stays are represented
sepsis_stays_in_labels = hourly_grid[hourly_grid['label']==1]['stay_id'].nunique()
print(f'\nSepsis stays with at least 1 positive label: {sepsis_stays_in_labels:,}')
print(f'Total sepsis stays: {len(sepsis_onset):,}')
print(f'Sepsis stays captured: {sepsis_stays_in_labels/len(sepsis_onset):.1%}')

# Save
hourly_grid.to_csv(OUTPUT_DIR / 'hourly_labels.csv', index=False)
print('\nhourly_labels.csv saved')

Hourly grid shape: (8179630, 4)

After filtering:
Total prediction points : 2,618,839
Positive labels         : 87,263
Negative labels         : 2,531,576
Positive rate           : 3.332%
Unique stays            : 74,550
Unique patients         : 55,559

Sepsis stays with at least 1 positive label: 26,643
Total sepsis stays: 45,823
Sepsis stays captured: 58.1%

hourly_labels.csv saved


## Understanding Sepsis breakdown ( Stays with no positive labels )

###### Reason 1 — Onset at hour 0 or hour 1 If sepsis onset is at hour 0 or 1, there is no valid prediction point before onset. You require hour >= 1 and exclude already-septic hours. A patient with onset at hour 1 has only hour 0 available before onset — which is excluded. These patients are uncapturable by design and that is correct — you cannot predict something that happens at admission.
###### Reason 2 — Stay too short If a patient's ICU stay ends before any valid prediction point exists before their onset, they get excluded.
###### Reason 3 — Onset exactly at hour boundary Minor edge cases where hours_to_onset computes to exactly 0.

In [ ]:
# Understand why 41.9% of sepsis stays have no positive labels
sepsis_in_labels = hourly_grid[hourly_grid['label'] == 1]['stay_id'].unique()
missing_sepsis = sepsis_onset[~sepsis_onset['stay_id'].isin(sepsis_in_labels)].copy()

# Merge with label_df to get onset hours
missing_sepsis = missing_sepsis.merge(
    label_df[['stay_id', 'onset_hour']],
    on='stay_id', how='left'
)

print(f"Sepsis stays with no positive labels: {len(missing_sepsis):,}")
print("\nOnset hour distribution for MISSING stays:")
print(missing_sepsis['onset_hour'].describe())
print("\nOnset at hour 0-1 (uncapturable by design):",
      (missing_sepsis['onset_hour'] < 1).sum())
print("Onset at hour 1-6 (very early, few prediction points):",
      ((missing_sepsis['onset_hour'] >= 1) &
       (missing_sepsis['onset_hour'] < 6)).sum())
print("Onset after hour 6:",
      (missing_sepsis['onset_hour'] >= 6).sum())

Sepsis stays with no positive labels: 19,180

Onset hour distribution for MISSING stays:
count    19180.000000
mean         0.474247
std          0.286540
min          0.000000
25%          0.229167
50%          0.463333
75%          0.716667
max          1.000000
Name: onset_hour, dtype: float64

Onset at hour 0-1 (uncapturable by design): 19137
Onset at hour 1-6 (very early, few prediction points): 43
Onset after hour 6: 0


In [ ]:
# Verify sepsis_onset is ready for Block 10
print("sepsis_onset shape:", sepsis_onset.shape)
print("sepsis_onset columns:", sepsis_onset.columns.tolist())
print("NaN in sepsis_onset_time:", sepsis_onset['sepsis_onset_time'].isna().sum())
print(sepsis_onset.head())

sepsis_onset shape: (45823, 2)
sepsis_onset columns: ['stay_id', 'sepsis_onset_time']
NaN in sepsis_onset_time: 0
    stay_id   sepsis_onset_time
0  30000484 2136-01-14 18:00:00
1  30000646 2194-04-29 06:00:00
2  30000831 2140-04-17 22:00:00
3  30001148 2156-08-30 14:00:00
4  30001396 2147-10-19 04:00:00


In [ ]:
# check cohort has outtime
print(cohort[['stay_id', 'subject_id', 'intime', 'outtime', 'icu_los_hours']].head())
print("outtime NaNs:", cohort['outtime'].isna().sum())
print("icu_los_hours NaNs:", cohort['icu_los_hours'].isna().sum())

    stay_id  subject_id              intime             outtime  icu_los_hours
0  39553978    10000032 2180-07-23 14:00:00 2180-07-23 23:50:47       9.846389
1  37081114    10000690 2150-11-02 19:37:00 2150-11-06 17:03:17      93.438056
2  39765666    10000980 2189-06-27 08:42:00 2189-06-27 20:38:27      11.940833
3  37067082    10001217 2157-11-20 19:18:02 2157-11-21 22:08:00      26.832778
4  34592300    10001217 2157-12-19 15:42:24 2157-12-20 14:27:41      22.754722
outtime NaNs: 0
icu_los_hours NaNs: 0


In [ ]:
# merge onset with ICU admission time
label_df = cohort[["stay_id", "intime"]].drop_duplicates().merge(
    sepsis_onset,
    on="stay_id",
    how="left"
)

label_df["onset_hour"] = (
    label_df["sepsis_onset_time"] - label_df["intime"]
).dt.total_seconds() / 3600.0

print(label_df.shape)
label_df.head()

(93730, 4)


,stay_id,intime,sepsis_onset_time,onset_hour
0,39553978,2180-07-23 14:00:00,NaT,NaN
1,37081114,2150-11-02 19:37:00,2150-11-02 20:00:00,0.383333
2,39765666,2189-06-27 08:42:00,2189-06-27 10:00:00,1.300000
3,37067082,2157-11-20 19:18:02,NaT,NaN
4,34592300,2157-12-19 15:42:24,NaT,NaN


In [ ]:
## Comparing with ICD

## Train/ Test Split

In [ ]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT — by subject_id
# Splitting by stay_id leaks patients across sets
# ============================================================
from sklearn.model_selection import train_test_split

unique_subjects = hourly_grid['subject_id'].unique()
print(f'Total unique patients: {len(unique_subjects):,}')

# 70% train, 15% val, 15% test
train_subjects, temp_subjects = train_test_split(
    unique_subjects, test_size=0.30, random_state=42
)
val_subjects, test_subjects = train_test_split(
    temp_subjects, test_size=0.50, random_state=42
)

train_stays = hourly_grid[hourly_grid['subject_id'].isin(train_subjects)]['stay_id'].unique()
val_stays   = hourly_grid[hourly_grid['subject_id'].isin(val_subjects)]['stay_id'].unique()
test_stays  = hourly_grid[hourly_grid['subject_id'].isin(test_subjects)]['stay_id'].unique()

print(f'\nTrain: {len(train_subjects):,} patients | {len(train_stays):,} stays')
print(f'Val:   {len(val_subjects):,} patients | {len(val_stays):,} stays')
print(f'Test:  {len(test_subjects):,} patients | {len(test_stays):,} stays')

# Verify zero leakage
assert len(set(train_stays) & set(test_stays)) == 0, "LEAKAGE: train/test stay overlap"
assert len(set(train_stays) & set(val_stays))  == 0, "LEAKAGE: train/val stay overlap"
assert len(set(val_stays)   & set(test_stays)) == 0, "LEAKAGE: val/test stay overlap"
print('\nNo leakage confirmed ✓')

# Positive rates per split — should be roughly equal
for name, subjects in [('Train', train_subjects),
                        ('Val',   val_subjects),
                        ('Test',  test_subjects)]:
    split = hourly_grid[hourly_grid['subject_id'].isin(subjects)]
    print(f'{name} | prediction points: {len(split):,} | '
          f'positive rate: {split["label"].mean():.3%}')

# Save split assignments
split_df = pd.DataFrame({
    'subject_id': list(train_subjects) + list(val_subjects) + list(test_subjects),
    'split':      (['train'] * len(train_subjects) +
                   ['val']   * len(val_subjects) +
                   ['test']  * len(test_subjects))
})
split_df.to_csv(OUTPUT_DIR / 'subject_splits.csv', index=False)
print('\nsubject_splits.csv saved')

# Save hourly_grid with split column attached for easy loading in File 3
hourly_grid = hourly_grid.merge(split_df, on='subject_id', how='left')
hourly_grid.to_csv(OUTPUT_DIR / 'hourly_labels.csv', index=False)
print('hourly_labels.csv updated with split column')

Total unique patients: 55,559

Train: 38,891 patients | 52,210 stays
Val:   8,334 patients | 11,097 stays
Test:  8,334 patients | 11,243 stays

No leakage confirmed ✓
Train | prediction points: 1,822,797 | positive rate: 3.384%
Val | prediction points: 396,334 | positive rate: 3.149%
Test | prediction points: 399,708 | positive rate: 3.277%

subject_splits.csv saved
hourly_labels.csv updated with split column
